# 03 — Stock-day FinBERT embeddings

Aggregate per-filing embeddings (from `02_finbert_embed_docs.ipynb`) into a `(permno, date)` panel via an exponentially-decay-weighted mean over a rolling 30-day window with 14-day half-life, then forward-fill days that had no recent filing (spec §5.2).

**Inputs**
- `data/processed/finbert_doc_embed/` — per-filing embeddings (notebook 02)
- `data/processed/panel/` — the unified PIT panel; gives the `(permno, date, cik)` grid (built by `src/data/build_panel.py`)

**Output**
- `data/processed/finbert_stockday_embed/year=YYYY/part-permno-NNNNNNNN.parquet` — rows `(permno, date, vec)` where `vec` is a 768-dim float32 list.

This dataset is the input to PCA reduction + the LightGBM ranker (spec §§5.3, 7).

## A. Setup

In [ ]:
from __future__ import annotations
import json
import shutil
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DOC_EMBED_DIR = REPO_ROOT / 'data' / 'processed' / 'finbert_doc_embed'
PANEL_DIR = REPO_ROOT / 'data' / 'processed' / 'panel'
OUT_DIR = REPO_ROOT / 'data' / 'processed' / 'finbert_stockday_embed'
SUMMARY_PATH = REPO_ROOT / 'reports' / 'metrics' / 'finbert_stockday_embed_summary.json'

WINDOW_DAYS = 30
HALFLIFE_DAYS = 14.0

print(f'window={WINDOW_DAYS}d, halflife={HALFLIFE_DAYS}d')
print(f'doc_embed_dir={DOC_EMBED_DIR}')
print(f'panel_dir={PANEL_DIR}')
print(f'out_dir={OUT_DIR}')

## B. Load document embeddings

Reads every year-shard from notebook 02 into a single in-memory frame. Each `vec` becomes a `(HIDDEN,)` float32 array.

In [ ]:
docs = duckdb.sql(f"SELECT cik, filing_date, vec FROM '{DOC_EMBED_DIR}/year=*/*.parquet'").df()
docs['filing_date'] = pd.to_datetime(docs['filing_date']).dt.normalize()
docs['vec'] = [np.asarray(v, dtype=np.float32) for v in docs['vec'].values]
HIDDEN = docs['vec'].iloc[0].shape[0]
print(f'doc embeddings: {len(docs):,} rows, hidden={HIDDEN}')
print(f'date range: {docs.filing_date.min().date()} -> {docs.filing_date.max().date()}')
print(f'unique ciks with >=1 filing: {docs.cik.nunique():,}')

## C. Load panel slim

Only `(permno, date, cik)` is needed to drive the aggregation. Panel `cik` is stored as `double` (legacy build); cast back to the 10-digit zero-padded string format that matches `edgar_index.cik` and our doc-embed `cik` column. Permno-days without a cik (no Sharadar coverage) are dropped — they couldn't get a text signal anyway.

In [ ]:
panel = duckdb.sql(f"SELECT permno, date, cik FROM '{PANEL_DIR}/year=*/*.parquet' WHERE cik IS NOT NULL").df()
panel['cik'] = panel['cik'].astype('int64').map(lambda x: f'{x:010d}')
panel['date'] = pd.to_datetime(panel['date']).dt.normalize()
panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)
print(f'panel rows: {len(panel):,}')
print(f'permnos:    {panel.permno.nunique():,}')
print(f'ciks:       {panel.cik.nunique():,}')
print(f'date range: {panel.date.min().date()} -> {panel.date.max().date()}')

## D. Per-permno aggregation + forward-fill

For each permno:
1. Find all doc embeddings whose `cik` ever matched the permno.
2. For every panel date `t`, compute `e_t = sum_j(w_j · vec_j) / sum_j(w_j)` over filings `j` with `0 ≤ t − filing_date_j ≤ 30`, where `w_j = 0.5 ** ((t − filing_date_j) / 14)`.
3. Forward-fill days where no filing fell in the 30-day window. Days before the very first in-window filing are dropped.

Dry run on 3 permnos to sanity-check shape + day counts.

In [ ]:
def aggregate_permno(panel_g: pd.DataFrame, docs: pd.DataFrame) -> pd.DataFrame:
    ciks = panel_g['cik'].unique()
    f = docs[docs['cik'].isin(ciks)].sort_values('filing_date')
    if f.empty:
        return pd.DataFrame(columns=['permno', 'date', 'vec'])
    fvecs = np.stack(f['vec'].values).astype(np.float32)
    fdates = f['filing_date'].values.astype('datetime64[D]')
    pdates = panel_g['date'].values.astype('datetime64[D]')
    days_lag = (pdates[:, None] - fdates[None, :]).astype('int64')
    in_window = (days_lag >= 0) & (days_lag <= WINDOW_DAYS)
    weights = np.where(in_window, 0.5 ** (days_lag / HALFLIFE_DAYS), 0.0).astype(np.float32)
    w_sum = weights.sum(axis=1)
    has_filing = w_sum > 0
    agg = (weights @ fvecs) / np.maximum(w_sum[:, None], 1e-12)
    agg = np.where(has_filing[:, None], agg, np.nan).astype(np.float32)
    last = None
    for i in range(agg.shape[0]):
        if has_filing[i]:
            last = agg[i]
        elif last is not None:
            agg[i] = last
    valid = ~np.isnan(agg[:, 0])
    if not valid.any():
        return pd.DataFrame(columns=['permno', 'date', 'vec'])
    return pd.DataFrame({
        'permno': panel_g['permno'].values[valid],
        'date': pdates[valid],
        'vec': [agg[i] for i in np.where(valid)[0]],
    })

for pn in panel['permno'].unique()[:3]:
    pg = panel[panel['permno'] == pn]
    out = aggregate_permno(pg, docs)
    if out.empty:
        print(f'permno={pn}: no overlap with doc embeddings')
        continue
    print(f"permno={pn}: panel_days={len(pg):,}, out_days={len(out):,}, first={out['date'].iloc[0]}, last={out['date'].iloc[-1]}")

## E. Full aggregation loop

One file per `(permno, year)` keeps per-permno memory bounded; DuckDB will glob-scan them downstream. A prior run is wiped before re-running so the output is a clean snapshot.

In [ ]:
SCHEMA = pa.schema([
    ('permno', pa.int64()),
    ('date', pa.timestamp('ns')),
    ('vec', pa.list_(pa.float32())),
])

def write_permno(out_df: pd.DataFrame, permno: int) -> int:
    if out_df.empty:
        return 0
    out_df = out_df.copy()
    out_df['year'] = pd.to_datetime(out_df['date']).dt.year
    total = 0
    for year, grp in out_df.groupby('year'):
        out_path = OUT_DIR / f'year={year}' / f'part-permno-{permno:08d}.parquet'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        records = [
            {'permno': int(r.permno), 'date': pd.Timestamp(r.date), 'vec': r.vec.tolist()}
            for r in grp.drop(columns='year').itertuples(index=False)
        ]
        table = pa.Table.from_pylist(records, schema=SCHEMA)
        pq.write_table(table, out_path, compression='zstd')
        total += len(records)
    return total

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

permnos = panel['permno'].unique()
written = 0
empty_permnos = 0
for pn in tqdm(permnos, desc='permnos'):
    pg = panel[panel['permno'] == pn]
    out = aggregate_permno(pg, docs)
    n = write_permno(out, int(pn))
    if n == 0:
        empty_permnos += 1
    written += n
print(f'\nwrote {written:,} stock-day rows across {len(permnos) - empty_permnos} permnos ({empty_permnos} had no doc overlap)')

## F. Validation + summary

In [ ]:
con = duckdb.connect()
total = con.sql(f"SELECT COUNT(*) FROM '{OUT_DIR}/year=*/*.parquet'").fetchone()[0]
permnos_out = con.sql(f"SELECT COUNT(DISTINCT permno) FROM '{OUT_DIR}/year=*/*.parquet'").fetchone()[0]
date_range = con.sql(f"SELECT MIN(date), MAX(date) FROM '{OUT_DIR}/year=*/*.parquet'").fetchone()
print(f'stock-day rows:   {total:,}')
print(f'permnos covered:  {permnos_out}')
print(f'date range:       {date_range[0]} -> {date_range[1]}')

sample = con.sql(f"SELECT vec FROM '{OUT_DIR}/year=*/*.parquet' USING SAMPLE 500 ROWS").df()
vecs = np.stack([np.asarray(v, dtype=np.float32) for v in sample['vec'].values])
print(f'sample shape: {vecs.shape}, nan: {np.isnan(vecs).any()}, inf: {np.isinf(vecs).any()}')
print(f'norm: mean={np.linalg.norm(vecs, axis=1).mean():.3f}, std={np.linalg.norm(vecs, axis=1).std():.3f}')

summary = {
    'total_stock_days': int(total),
    'permnos_covered': int(permnos_out),
    'date_range': [str(date_range[0]), str(date_range[1])],
    'hidden_dim': int(HIDDEN),
    'window_days': WINDOW_DAYS,
    'halflife_days': HALFLIFE_DAYS,
    'doc_embed_dir': str(DOC_EMBED_DIR.relative_to(REPO_ROOT)),
    'output_dir': str(OUT_DIR.relative_to(REPO_ROOT)),
}
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
print(f'summary -> {SUMMARY_PATH.relative_to(REPO_ROOT)}')